# Integrating Synthetic Data into a full ML Ops end-to-end system using SageMaker Pipelines service 



In [2]:
import boto3
import sagemaker


region = boto3.Session().region_name
role = sagemaker.get_execution_role()
default_bucket = sagemaker.session.Session().default_bucket()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


In [22]:
import yaml

with open("configs/config_churn.yaml", 'r') as config_file:
    config = yaml.safe_load(config_file)

print(yaml.dump(config))

ML:
  ml_eval_metric: f1
  ml_metric_threshold: 0.0
  ml_task: classification
  objective: binary:logistic
  objective_type: Maximize
dataset:
  drop_columns: CustomerId
  name: churn
  path: s3://gretel-public-website/datasets/Churn_Modelling.csv
  target_column: Exited
gretel:
  generate_factor: 1.0
  strategy: augment
  target_balance: 1.0



In [21]:
!ls configs/config_churn.yaml

config_churn.yaml


### Get the pipeline instance

Here we get the pipeline instance from your pipeline module so that we can work with it.

In [8]:
from pipelines.scripts.pipeline import get_pipeline
from pipelines.scripts.datasets import datasets

# Change these to reflect your project/business name or if you want to separate ModelPackageGroup/Pipeline from the rest of your team
model_package_group_name = f"GretelModelPackageGroup-{config['dataset']['name']}"
pipeline_name = f"GretelPipeline-{config['dataset']['name']}"

pipeline = get_pipeline(
    region=region,
    role=role,
    default_bucket=default_bucket,
    model_package_group_name=model_package_group_name,
    pipeline_name=pipeline_name,
    config=config,
    use_gretel=True
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.xlarge.


### Submit the pipeline to SageMaker and start execution

Let's submit our pipeline definition to the workflow service. The role passed in will be used by the workflow service to create all the jobs defined in the steps.

In [6]:
pipeline.upsert(role_arn=role)

INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/0aa6a20d5c0ab701164c2ea150adfb25/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/d740a32944cbfc42a3149407782d0920/runproc.sh


Using provided s3_resource
Using provided s3_resource


INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/e1f60b1e75e14270d64dab25860a6efc/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/b724fc16e3228521d3bb13c708323145/runproc.sh
INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/0aa6a20d5c0ab701164c2ea150adfb25/sourcedir.tar.gz


Using provided s3_resource


INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/d740a32944cbfc42a3149407782d0920/runproc.sh


Using provided s3_resource


INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/e1f60b1e75e14270d64dab25860a6efc/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-dataset-stroke-data/code/b724fc16e3228521d3bb13c708323145/runproc.sh


{'PipelineArn': 'arn:aws:sagemaker:us-east-1:524473328983:pipeline/GretelPipeline-healthcare-dataset-stroke-data',
 'ResponseMetadata': {'RequestId': '15c56f64-a836-4938-bf79-6583d65e0a6d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '15c56f64-a836-4938-bf79-6583d65e0a6d',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '113',
   'date': 'Fri, 20 Oct 2023 04:56:43 GMT'},
  'RetryAttempts': 0}}

We'll start the pipeline, accepting all the default parameters.

Values can also be passed into these pipeline parameters on starting of the pipeline, and will be covered later. 

In [7]:
execution = pipeline.start()

### Pipeline Operations: examining and waiting for pipeline execution

Now we describe execution instance and list the steps in the execution to find out more about the execution.

In [7]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:524473328983:pipeline/GretelPipeline-healthcare-dataset-stroke-data',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:524473328983:pipeline/GretelPipeline-healthcare-dataset-stroke-data/execution/tqso2hfexygq',
 'PipelineExecutionDisplayName': 'execution-1697776191713',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2023, 10, 20, 4, 29, 51, 641000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2023, 10, 20, 4, 29, 51, 641000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:524473328983:user-profile/d-ktsm4iude3d4/gretel-sagemaker-mlops',
  'UserProfileName': 'gretel-sagemaker-mlops',
  'DomainId': 'd-ktsm4iude3d4'},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:524473328983:user-profile/d-ktsm4iude3d4/gretel-sagemaker-mlops',
  'UserProfileName': 'gretel-sagemaker-mlops',
  'DomainId': 'd-ktsm4iude3d4'},
 'ResponseMetadata': {'RequestId': 'a4

We can wait for the execution by invoking `wait()` on the execution:

In [8]:
execution.wait()

KeyboardInterrupt: 

We can list the execution steps to check out the status and artifacts:

In [ ]:
execution.list_steps()

In [10]:
# clean-up 

client = boto3.client('sagemaker')
pl_list = client.list_pipelines()
for pl in pl_list['PipelineSummaries']:
    print(f"Removing {pl['PipelineName']}")
    client.delete_pipeline(PipelineName=pl['PipelineName'])

Removing GretelPipeline-healthcare-dataset-stroke-data


In [12]:
import yaml

data = {
    "dataset": {
        "name": "churn",
        "path": "s3://gretel-public-website/datasets/Churn_Modelling.csv",
        "label_column_name": "Exited",
        "drop_columns": "CustomerId",
    },
    "gretel": {
        "strategy": "augment",
        "generate_factor": 1.0,
        "target_balance": 1.0
    },
    "ML": {
        "objective": "binary:logistic",
        "objective_type": "Maximize",
        "ml_eval_metric": "f1",
        "ml_task": "classification",
        "ml_metric_threshold": 0.0
    }
}

# Define the file path where you want to save the YAML file
yaml_file_path = "config.yaml"

# Write the dictionary to a YAML file
with open(yaml_file_path, 'w') as yaml_file:
    yaml.dump(data, yaml_file, default_flow_style=False)

print(f"The YAML file '{yaml_file_path}' has been created.")


The YAML file 'config.yaml' has been created.


In [13]:
with open(yaml_file_path, 'r') as yaml_file:
    yaml_dict = yaml.safe_load(yaml_file)

In [15]:
yaml_dict['dataset']['name']

'churn'

In [10]:
# Create a dictionary with column names and data types
column_data_types = {}
for column_name, data_type in zip(df.columns, df.dtypes):
    # Map pandas data types to numpy data types
    numpy_data_type = np.dtype(data_type).name
    column_data_types[column_name] = numpy_data_type

# Format the dictionary as required
feature_columns = {
    "feature_columns": column_data_types
}

print(feature_columns)

{'feature_columns': {'id': 'int64', 'gender': 'object', 'age': 'float64', 'hypertension': 'int64', 'heart_disease': 'int64', 'ever_married': 'object', 'work_type': 'object', 'Residence_type': 'object', 'avg_glucose_level': 'float64', 'bmi': 'float64', 'smoking_status': 'object', 'stroke': 'int64'}}


In [14]:
def merge_two_dicts(x, y):
    """Merges two dicts, returning a new copy."""
    z = x.copy()
    z.update(y)
    return z

feature_columns= {
    "sex": str,
    "length": np.float64,
    "diameter": np.float64,
    "height": np.float64,
    "whole_weight": np.float64,
    "shucked_weight": np.float64,
    "viscera_weight": np.float64,
    "shell_weight": np.float64,
}
label_column= {
    "rings": np.float64
}

In [10]:
import pandas as pd
df = pd.read_csv(
    "s3://gretel-public-website/datasets/kaggle_datasets_ey/Churn_Modelling.csv",
)

In [11]:
list(df.columns)

['RowNumber',
 'CustomerId',
 'Surname',
 'CreditScore',
 'Geography',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited']

In [21]:
# Identify categorical columns
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(categorical_columns)


['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
